In [ ]:
!pip install opencv-python
!pip install yt_dlp

In [ ]:
import cv2
import os
import yt_dlp
import time
from google.colab import drive

# 1. Connexion au Drive (sans afficher l'erreur si c'est déjà connecté)
try:
    drive.mount('/content/drive')
except:
    pass

DOSSIER_BASE_DRIVE = '/content/drive/MyDrive/Dataset_Kaggle'

def extraire_dataset(categorie, nom_du_jeu, url_youtube, intervalle_secondes=5):
    dossier_sortie = os.path.join(DOSSIER_BASE_DRIVE, categorie)
    os.makedirs(dossier_sortie, exist_ok=True)

    fichier_video_temp = f"temp_{nom_du_jeu}.mp4"

    # CORRECTION ICI : On force un format lisible par OpenCV (H.264 / avc1)
    ydl_opts = {
        'format': 'bestvideo[ext=mp4][vcodec^=avc1][height<=720]/best[ext=mp4][height<=720]',
        'outtmpl': fichier_video_temp,
        'quiet': False,
        'noplaylist': True,
    }

    print(f"\n--- 📥 ÉTAPE 1 : Téléchargement de la vidéo pour {nom_du_jeu} ---")

    try:
        with yt_dlp.YoutubeDL(ydl_opts) as ydl:
            ydl.download([url_youtube])
    except Exception as e:
        print(f"❌ Erreur lors du téléchargement : {e}")
        return

    print(f"\n--- 🎞️ ÉTAPE 2 : Extraction des images avec OpenCV ---")
    cap = cv2.VideoCapture(fichier_video_temp)
    if not cap.isOpened():
        print(f"❌ Impossible d'ouvrir la vidéo. Le fichier est peut-être corrompu.")
        return

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0: fps = 30

    frame_interval = int(fps * intervalle_secondes)
    count = 0
    saved_count = 0

    print(f"Découpage en cours (1 image toutes les {intervalle_secondes} secondes)...")
    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        # Recadrage anti-bandes noires
        frame = frame[10:-10, 10:-10]

        if count % frame_interval == 0:
            nom_fichier = os.path.join(dossier_sortie, f"{nom_du_jeu}_{saved_count:04d}.jpg")
            cv2.imwrite(nom_fichier, frame)
            saved_count += 1

            if saved_count % 50 == 0:
                print(f"📸 {saved_count} images générées et sauvegardées sur le Drive...")

        count += 1

    cap.release()

    # Nettoyage
    if os.path.exists(fichier_video_temp):
        os.remove(fichier_video_temp)

    # Vérification finale
    if saved_count == 0:
        print(f"⚠️ AVERTISSEMENT : La vidéo a été téléchargée mais OpenCV n'a extrait aucune image.")
    else:
        print(f"✅ Terminé pour {nom_du_jeu} ! {saved_count} images ajoutées dans le dossier '{categorie}'.")

# 🎮 NOUVELLE LISTE SUPPLÉMENTAIRE (Ultra-Clean, World of Longplays)

liste_videos = [

]

# Lancement global
for item in liste_videos:
    extraire_dataset(item["categorie"], item["jeu"], item["url"], intervalle_secondes=20)
    time.sleep(2)

In [ ]:
# 1. Mount Drive
from google.colab import drive
drive.mount('/content/drive')

# 2. Install deps
!pip install -q kaggle

# 3. Setup avec le nouveau format de token
import os, json
# Token Kaggle : NE JAMAIS l'ecrire en dur ici (le notebook part sur git !).
# Sur Colab : icone cle (Secrets) > Ajouter un secret nomme KAGGLE_API_TOKEN,
# puis on le lit de maniere securisee :
from google.colab import userdata
os.environ['KAGGLE_API_TOKEN'] = userdata.get('KAGGLE_API_TOKEN')

# 4. Init metadata directement dans Drive
dataset_path = '/content/drive/MyDrive/Dataset_Kaggle'
!kaggle datasets init -p "{dataset_path}"

meta_path = f'{dataset_path}/dataset-metadata.json'
with open(meta_path) as f:
    meta = json.load(f)

meta['id'] = 'maximeclment/datasetpaia3'
meta['title'] = 'PA IA3ABD2 Videogame Screenshots'

with open(meta_path, 'w') as f:
    json.dump(meta, f)

# 5. Push
!kaggle datasets version -p "{dataset_path}" -m "Add new images" --dir-mode zip